### Project 1: Customer Segmentation using RFM Analysis

### 1. Import Libraries 

In [2]:
import pandas as pd 
import numpy as np

SQL Data Preparation 

SELECT
customer_email,

COUNT(DISTINCT order_id) AS total_orders,

SUM(revenue) AS total_revenue,

MAX(order_date) AS last_purchase_date

FROM orders

GROUP BY 1


I used SQL to aggregate transaction-level purchase data into a customer-level dataset with order counts, revenue, and last purchase date. I then used Python with Pandas to calculate Recency, Frequency, and Monetary metrics and convert them into RFM scores. Based on those scores, I classified customers into segments such as high-value, loyal, promising, and churn-risk cohorts

### 2. Load the Dataset 

In [9]:
df = pd.read_csv("../data/wmg_rfm_orders_practice.csv")
df.head()

,order_id,order_date,customer_email,artist_name,store_name,product_name,product_type,units,revenue,true_segment_for_learning
0,100000,2024-08-06,fan00001@example.com,Coldplay,US Store,CD,music,1,32.98,Loyal
1,100000,2024-08-06,fan00001@example.com,Ed Sheeran,EU Store,Tote Bag,merch,1,32.98,Loyal
2,100001,2024-09-19,fan00001@example.com,Ed Sheeran,US Store,Cap,merch,1,5.00,Loyal
3,100001,2024-09-19,fan00001@example.com,My Chemical Romance,US Store,Vinyl,music,1,39.24,Loyal
4,100001,2024-09-19,fan00001@example.com,Twenty One Pilots,CA Store,T-Shirt,merch,1,17.93,Loyal


Our dataset is order-level data.

That means:

one row = one purchased item

the same customer can appear many times

the same order can also appear multiple times if the order had multiple products

### 3. Check the Shape of the Data 

In [13]:
df.shape 

# This tells us: number of rows, number of columns

(101046, 10)

### 4 Columns 

In [15]:
df.columns 

#This shows us the names of columns 

Index(['order_id', 'order_date', 'customer_email', 'artist_name', 'store_name',
       'product_name', 'product_type', 'units', 'revenue',
       'true_segment_for_learning'],
      dtype='object')

### 5. Inspect the Data Types

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101046 entries, 0 to 101045
Data columns (total 10 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   order_id                   101046 non-null  int64  
 1   order_date                 101046 non-null  object 
 2   customer_email             101046 non-null  object 
 3   artist_name                101046 non-null  object 
 4   store_name                 101046 non-null  object 
 5   product_name               101046 non-null  object 
 6   product_type               101046 non-null  object 
 7   units                      101046 non-null  int64  
 8   revenue                    101046 non-null  float64
 9   true_segment_for_learning  101046 non-null  object 
dtypes: float64(1), int64(2), object(7)
memory usage: 7.7+ MB


This tells us:

which columns are text

which are numbers

whether there are null values

Very important: order_date is often loaded as text first, so we usually convert it.

### 6. Convert Order Date to Datetime 


In [17]:
df['order_date'] = pd.to_datetime(df["order_date"])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101046 entries, 0 to 101045
Data columns (total 10 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   order_id                   101046 non-null  int64         
 1   order_date                 101046 non-null  datetime64[ns]
 2   customer_email             101046 non-null  object        
 3   artist_name                101046 non-null  object        
 4   store_name                 101046 non-null  object        
 5   product_name               101046 non-null  object        
 6   product_type               101046 non-null  object        
 7   units                      101046 non-null  int64         
 8   revenue                    101046 non-null  float64       
 9   true_segment_for_learning  101046 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(6)
memory usage: 7.7+ MB


Dates need to be proper date format.

Why? Because later we want to calculate:

1. last purchase date
2. recency
3. time differences

If order_date stays as text, Python cannot do date math properly.

### 7. View a few example rows 

In [19]:
df.sample(10, random_state=43)

,order_id,order_date,customer_email,artist_name,store_name,product_name,product_type,units,revenue,true_segment_for_learning
17139,108609,2024-09-15,fan01670@example.com,Twenty One Pilots,EU Store,Hoodie,merch,1,40.64,Loyal
77818,138915,2024-04-14,fan07705@example.com,Twenty One Pilots,EU Store,Cap,merch,1,73.86,High Value
20999,110551,2023-12-16,fan02069@example.com,My Chemical Romance,US Store,Vinyl,music,1,5.00,High Value
98140,149003,2024-10-31,fan09712@example.com,Dua Lipa,US Store,Cassette,music,1,5.00,High Value
84583,142286,2023-10-23,fan08379@example.com,Ed Sheeran,UK Store,Hoodie,merch,1,16.43,High Value
68303,134159,2024-03-30,fan06729@example.com,Coldplay,CA Store,CD,music,1,5.00,High Value
92633,146275,2024-09-05,fan09163@example.com,Dua Lipa,US Store,Cassette,music,1,11.21,Occasional
10313,105197,2023-09-20,fan00987@example.com,My Chemical Romance,CA Store,Cassette,music,1,21.82,Churn Risk
46969,123510,2023-10-15,fan04638@example.com,Twenty One Pilots,US Store,T-Shirt,merch,1,60.05,High Value
75874,137939,2024-07-13,fan07508@example.com,Ed Sheeran,UK Store,CD,music,1,5.00,High Value


This gives us a random sample of rows.

Why do this?

Because when learning analytics, it is very important to actually look at the data.

You want to notice things like:

does one customer appear multiple times?

does one order contain multiple products?

do the revenue numbers look sensible?

### 8. Count unique customers and orders

In [22]:
num_customers = df["customer_email"].nunique()
num_orders = df["order_id"].nunique()

print("Unique customers:", num_customers)
print("Unique orders:", num_orders)

Unique customers: 10000
Unique orders: 50465


This tells us:

how many customers are in the data

how many orders are in the data

This is important because the raw table is not yet customer-level.

Right now we have transaction/item-level rows.

### 9. Understand the Business Problem

In [25]:
print("Business Problem:")
print("Marketing is treating all customers the same.")
print("I want to identify high-value, loyal, occasional, and churn-risk customers.")

Business Problem:
Marketing is treating all customers the same.
I want to identify high-value, loyal, occasional, and churn-risk customers.


This is the core business problem for Project 1.

Before segmentation:

marketing sends campaigns too broadly

no clear prioritization

high-value and churn-risk fans are mixed together

My proposal is:

create customer segments based on behavior

help marketing target smarter

In [ ]:
10. Why raw data is not enough 

In [27]:
df[["order_id", "customer_email", "order_date", "revenue"]].head(10)

,order_id,customer_email,order_date,revenue
0,100000,fan00001@example.com,2024-08-06,32.98
1,100000,fan00001@example.com,2024-08-06,32.98
2,100001,fan00001@example.com,2024-09-19,5.00
3,100001,fan00001@example.com,2024-09-19,39.24
4,100001,fan00001@example.com,2024-09-19,17.93
5,100002,fan00001@example.com,2024-11-01,67.77
6,100003,fan00002@example.com,2024-04-24,13.16
7,100003,fan00002@example.com,2024-04-24,13.29
8,100003,fan00002@example.com,2024-04-24,23.79
9,100004,fan00002@example.com,2024-07-11,30.31


This helps you see the problem.

A single customer may appear in multiple rows because they made multiple purchases.

So if we want segmentation, we cannot stay at row level.

We must move to:

one row per customer

That is the most important structural step in this project.

In [ ]:
11. Build a Customer Level Summary Table 

In [30]:
customer_df = (
    df.groupby("customer_email")
    .agg(
    total_orders=("order_id", "nunique"),
    total_revenue=("revenue","sum"), 
    last_purchase_date=("order_date", "max")
    )
    .reset_index()
)
    
customer_df.head()

,customer_email,total_orders,total_revenue,last_purchase_date
0,fan00001@example.com,3,195.90,2024-11-01
1,fan00002@example.com,2,80.55,2024-07-11
2,fan00003@example.com,3,87.56,2024-09-15
3,fan00004@example.com,10,914.07,2024-11-09
4,fan00005@example.com,3,154.72,2024-08-17


Explanation

This is the heart of the project.

We are grouping the data by customer and creating one row per customer.

What each part means

groupby("customer_email")

group all rows belonging to the same customer together

total_orders=("order_id", "nunique")

count unique orders for each customer

total_revenue=("revenue", "sum")

total amount spent by that customer

last_purchase_date=("order_date", "max")

latest purchase date for that customer

After this step, each row represents one customer.

In [ ]:
12. Check the customer_level dataset

In [31]:
customer_df.shape

(10000, 4)

Now the number of rows should be equal to the number of customers.

This confirms that we successfully moved from:

order/item-level data
to

customer-level data

### 13. Look at Sample Customers

In [32]:
customer_df.sample(10, random_state=42)

,customer_email,total_orders,total_revenue,last_purchase_date
6252,fan06253@example.com,1,28.31,2024-06-08
4684,fan04685@example.com,2,40.52,2023-10-21
1731,fan01732@example.com,10,891.78,2024-11-04
4742,fan04743@example.com,5,291.41,2024-11-08
4521,fan04522@example.com,2,51.92,2023-09-18
6340,fan06341@example.com,3,186.31,2024-08-14
576,fan00577@example.com,5,327.75,2024-08-10
5202,fan05203@example.com,10,886.58,2024-10-24
6363,fan06364@example.com,6,551.37,2024-10-25
439,fan00440@example.com,2,54.94,2024-03-17


Now you can actually see the kind of table RFM needs.

Each customer now has:

order count

total spend

last purchase date

But we still need to calculate recency.

### 14. Choose an Analysis Date 

In [37]:
df["order_date"] = pd.to_datetime(df["order_date"])
analysis_date = df["order_date"].max() + pd.Timedelta(days=1)
analysis_date

Timestamp('2024-12-01 00:00:00')

Explanation: We need a reference date for calculating recency.

Recency means: how many days since the customer last purchased

A common choice is: one day after the latest date in the dataset

Why? Because that makes recency logical and consistent.

Example: latest order date in dataset = 2024-12-31
analysis date = 2025-01-01

Then: someone who bought on 2024-12-30 has recency = 2 days

In [ ]:
15. Calculate Recency

In [38]:
customer_df["last_purchase_date"] = pd.to_datetime(customer_df["last_purchase_date"])
customer_df["recency_days"] = (analysis_date - customer_df["last_purchase_date"]).dt.days
customer_df.head(2)

,customer_email,total_orders,total_revenue,last_purchase_date,recency_days
0,fan00001@example.com,3,195.90,2024-11-01,30
1,fan00002@example.com,2,80.55,2024-07-11,143


recency = analysis_date - last_purchase_date 

Important interpretation:

lower recency = customer bought recently = better

higher recency = customer has been inactive for longer = worse

In [ ]:
16. Rename Columns for RFM Clarity 

In [39]:
customer_df = customer_df.rename(
columns={
    "total_orders":"frequency", 
    "total_revenue":"monetary"
})

customer_df.head()

,customer_email,frequency,monetary,last_purchase_date,recency_days
0,fan00001@example.com,3,195.90,2024-11-01,30
1,fan00002@example.com,2,80.55,2024-07-11,143
2,fan00003@example.com,3,87.56,2024-09-15,77
3,fan00004@example.com,10,914.07,2024-11-09,22
4,fan00005@example.com,3,154.72,2024-08-17,106


This just makes the names match RFM language.

Now we have:

recency_days

frequency

monetary

That makes the project easier to explain in interviews.

### 17. Inspect Summary Statistics 

In [40]:
customer_df[["recency_days", "frequency","monetary"]].describe()

,recency_days,frequency,monetary
count,10000.000000,10000.000000,10000.000000
mean,110.424100,5.046500,397.000892
std,113.649331,3.378503,368.114382
min,1.000000,1.000000,8.670000
25%,29.000000,2.000000,67.507500
50%,66.000000,4.000000,259.945000
75%,157.000000,7.000000,711.322500
max,450.000000,12.000000,1320.710000


### 18. Create R Score 

In [43]:
customer_df["R_score"] = pd.qcut(
    customer_df["recency_days"],
    5,
    labels=[5, 4, 3, 2, 1]
)

customer_df.head()

,customer_email,frequency,monetary,last_purchase_date,recency_days,R_score
0,fan00001@example.com,3,195.90,2024-11-01,30,4
1,fan00002@example.com,2,80.55,2024-07-11,143,2
2,fan00003@example.com,3,87.56,2024-09-15,77,3
3,fan00004@example.com,10,914.07,2024-11-09,22,5
4,fan00005@example.com,3,154.72,2024-08-17,106,2


We are creating a score from 1 to 5 for recency.

Why use qcut?

pd.qcut() divides the customers into equal-sized groups.

If we use 5 groups, that means:

top 20%

next 20%

next 20%

next 20%

bottom 20%

Why labels [5, 4, 3, 2, 1]?

Because for recency:

lower recency is better

recent customers should get a higher score

So the most recent customers get 5.

### 19. Create F Score

In [45]:
customer_df["F_score"] = pd.qcut(
customer_df["frequency"].rank(method="first"), 
5, 
labels=[1,2,3,4,5]
)

customer_df.head(2)

,customer_email,frequency,monetary,last_purchase_date,recency_days,R_score,F_score
0,fan00001@example.com,3,195.90,2024-11-01,30,4,2
1,fan00002@example.com,2,80.55,2024-07-11,143,2,1


This creates the frequency score.

Why use .rank(method="first")?

Because many customers may have the same frequency, and qcut can struggle with duplicates.

Ranking first gives each customer a unique position.

For frequency:

more orders = better

so higher frequency gets higher score

### 20. Create M score 

In [49]:
customer_df["M_Score"] = pd.qcut(
customer_df["monetary"].rank(method="first"),
5,
labels=[1,2,3,4,5])

customer_df.head(2)

,customer_email,frequency,monetary,last_purchase_date,recency_days,R_score,F_score,M_Score
0,fan00001@example.com,3,195.90,2024-11-01,30,4,2,3
1,fan00002@example.com,2,80.55,2024-07-11,143,2,1,2


### 21. Convert Scores to Integers

In [52]:
customer_df["R_score"] = customer_df["R_score"].astype(int)
customer_df["F_score"] = customer_df["F_score"].astype(int)
customer_df["M_score"] = customer_df["M_Score"].astype(int)

customer_df.head()

,customer_email,frequency,monetary,last_purchase_date,recency_days,R_score,F_score,M_Score,M_score
0,fan00001@example.com,3,195.90,2024-11-01,30,4,2,3,3
1,fan00002@example.com,2,80.55,2024-07-11,143,2,1,2,2
2,fan00003@example.com,3,87.56,2024-09-15,77,3,2,2,2
3,fan00004@example.com,10,914.07,2024-11-09,22,5,5,5,5
4,fan00005@example.com,3,154.72,2024-08-17,106,2,2,2,2


By default, qcut labels may behave like categories.

Converting to integers makes them easier to use in logic and calculations.

### 22. Create RFM Combined Code

In [53]:
customer_df["RFM_code"] = (
    customer_df["R_score"].astype(str)
    + customer_df["F_score"].astype(str)
    + customer_df["M_score"].astype(str)
)

customer_df.head()

,customer_email,frequency,monetary,last_purchase_date,recency_days,R_score,F_score,M_Score,M_score,RFM_code
0,fan00001@example.com,3,195.90,2024-11-01,30,4,2,3,3,423
1,fan00002@example.com,2,80.55,2024-07-11,143,2,1,2,2,212
2,fan00003@example.com,3,87.56,2024-09-15,77,3,2,2,2,322
3,fan00004@example.com,10,914.07,2024-11-09,22,5,5,5,5,555
4,fan00005@example.com,3,154.72,2024-08-17,106,2,2,2,2,222


This combines the three scores into one code.

Example:

R = 5

F = 4

M = 5

Then RFM code = "545"

This is useful for analysis, though the business usually prefers readable segment names.

In [ ]:
### 23. Define Segment Rules 

In [58]:
def assign_segment(row):
    r = row["R_score"]
    f = row["F_score"]
    m = row["M_score"]
    
    if r >= 4 and f >= 4 and m >= 4:
        return "High Value"
    elif r <= 2:
        return "Churn Risk"
    elif r >= 4 and f <= 2:
        return "Promising"
    elif r >= 3 and f >= 3:
        return "Loyal"
    else:
        return "Occasional"
    

This is the business logic for final customer segments.

There is no single universal rule in the world.

Companies define segments in ways that make sense for their business.

Here we use beginner-friendly rules:

High Value = recent, frequent, high spend

Churn Risk = poor recency

Promising = recent but not yet frequent

Loyal = decent recency and frequency

Occasional = everyone else

### 24. Apply the Segment Logic

In [59]:
customer_df["segment"] = customer_df.apply(assign_segment, axis=1)
customer_df.head()

,customer_email,frequency,monetary,last_purchase_date,recency_days,R_score,F_score,M_Score,M_score,RFM_code,segment
0,fan00001@example.com,3,195.90,2024-11-01,30,4,2,3,3,423,Promising
1,fan00002@example.com,2,80.55,2024-07-11,143,2,1,2,2,212,Churn Risk
2,fan00003@example.com,3,87.56,2024-09-15,77,3,2,2,2,322,Occasional
3,fan00004@example.com,10,914.07,2024-11-09,22,5,5,5,5,555,High Value
4,fan00005@example.com,3,154.72,2024-08-17,106,2,2,2,2,222,Churn Risk


This applies the function row by row.

Now every customer gets a segment label.

This is the final outcome of the segmentation project.

### 25. View Segment Cohorts 

In [61]:
segment_counts = customer_df["segment"].value_counts()
segment_counts

segment
Churn Risk    3954
High Value    3437
Loyal         1770
Occasional     710
Promising      129
Name: count, dtype: int64

This applies the function row by row.

Now every customer gets a segment label.

This is the final outcome of the segmentation project.

### 26 View Segment Percentages

In [62]:
segment_pct = (
    customer_df["segment"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

segment_pct

segment
Churn Risk    39.5
High Value    34.4
Loyal         17.7
Occasional     7.1
Promising      1.3
Name: proportion, dtype: float64

This converts counts into percentages.

This matters because in interviews you usually say things like:

“About 35% of customers were high value”

“About 18% were churn risk”

That is much more meaningful than just raw counts.

In [63]:
segment_summary = (
    customer_df.groupby("segment")
    .agg(
        customers=("customer_email", "count"),
        avg_recency=("recency_days", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_monetary=("monetary", "mean")
    )
    .reset_index()
)

segment_summary

,segment,customers,avg_recency,avg_frequency,avg_monetary
0,Churn Risk,3954,220.784269,2.267071,98.509052
1,High Value,3437,22.838522,8.947629,846.708342
2,Loyal,1770,56.581356,4.951412,322.643045
3,Occasional,710,68.257746,2.249296,106.835746
4,Promising,129,32.186047,3.000000,181.674341


This helps you validate the segments.

For example:

High Value should have:

lower recency

higher frequency

higher monetary

Churn Risk should have:

higher recency

lower activity

This is important because interviewers may ask:

“How did you validate that your segmentation made sense?”

This summary table is one good answer.

### Sort Summary Table

In [64]:
segment_summary = segment_summary.sort_values(by="avg_monetary", ascending=False)
segment_summary

,segment,customers,avg_recency,avg_frequency,avg_monetary
1,High Value,3437,22.838522,8.947629,846.708342
2,Loyal,1770,56.581356,4.951412,322.643045
4,Promising,129,32.186047,3.000000,181.674341
3,Occasional,710,68.257746,2.249296,106.835746
0,Churn Risk,3954,220.784269,2.267071,98.509052


In [65]:
print("Percent High Value:",
      round((customer_df["segment"] == "High Value").mean() * 100, 1), "%")

print("Percent Churn Risk:",
      round((customer_df["segment"] == "Churn Risk").mean() * 100, 1), "%")

Percent High Value: 34.4 %
Percent Churn Risk: 39.5 %


### What Marketing Would do with This

In [66]:
marketing_actions = {
    "High Value": "VIP drops, exclusive bundles, early access campaigns",
    "Loyal": "New release alerts, fan loyalty campaigns",
    "Promising": "Upsell and second-purchase encouragement",
    "Occasional": "General nurture campaigns",
    "Churn Risk": "Reactivation emails, discount offers"
}

for seg, action in marketing_actions.items():
    print(f"{seg}: {action}")

High Value: VIP drops, exclusive bundles, early access campaigns
Loyal: New release alerts, fan loyalty campaigns
Promising: Upsell and second-purchase encouragement
Occasional: General nurture campaigns
Churn Risk: Reactivation emails, discount offers


This is the bridge from analytics to business action.

Segmentation is only useful if someone uses it.

So your business story should be:

analysis created the customer groups

marketing used those groups differently